In [ ]:
# ==========================================
# OPENALEX: поиск статей с PDF
# + краткое содержание НА РУССКОМ
# + выбор чекбоксами
# + скачивание выбранных статей
# Для Jupyter Notebook
# ==========================================
from pathlib import Path
from typing import List, Dict, Optional
import re
import json
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# ------------------------------------------
# НАСТРОЙКИ
# ------------------------------------------

OPENALEX_WORKS_URL = "https://api.openalex.org/works"
USER_AGENT = "OpenAlexPaperFinder/1.0 (mailto:andrefan1406@gmail.com)"
REQUEST_TIMEOUT = 60
MAX_RESULTS_DEFAULT = 20
DOWNLOAD_CHUNK_SIZE = 1024 * 128

# ---- LLM / Ollama ----
OLLAMA_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "gpt-oss:120b-cloud"   # при необходимости замени


# ------------------------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ------------------------------------------

def _safe_get(d: dict, *keys, default=None):
    """
    Безопасное получение вложенных полей.
    """
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def _reconstruct_abstract(abstract_inverted_index: Optional[dict]) -> str:
    """
    Восстанавливает abstract из abstract_inverted_index OpenAlex.
    """
    if not abstract_inverted_index or not isinstance(abstract_inverted_index, dict):
        return ""

    positions = []
    for word, inds in abstract_inverted_index.items():
        for i in inds:
            positions.append((i, word))

    if not positions:
        return ""

    positions.sort(key=lambda x: x[0])
    tokens = [word for _, word in positions]
    text = " ".join(tokens)

    # Простая очистка пробелов перед пунктуацией
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()


def _guess_language(text: str) -> str:
    """
    Простая эвристика, если language не пришёл из OpenAlex.
    """
    if not text:
        return "unknown"

    sample = text[:3000]
    cyr = len(re.findall(r"[А-Яа-яЁё]", sample))
    lat = len(re.findall(r"[A-Za-z]", sample))

    if cyr > lat * 1.5:
        return "ru"
    if lat > cyr * 1.5:
        return "en"
    if cyr > 0 and lat > 0:
        return "mixed"
    return "unknown"


def _translate_to_ru(text: str) -> str:
    """
    Переводит текст на русский через Ollama / gpt-oss.
    Если перевод не удался — возвращает исходный текст.
    """
    if not text:
        return ""

    prompt = f"""
Переведи следующий текст на русский язык.
Сохрани смысл, не добавляй ничего лишнего.
Пиши только перевод.

Текст:
{text}
""".strip()

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": LLM_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json().get("response", "").strip()

    except Exception as e:
        print("⚠️ Ошибка перевода:", e)
        return text


def _build_short_summary(title: str, abstract_text: str, max_sentences: int = 3) -> str:
    """
    Делает краткое описание статьи.
    КРАТКОЕ СОДЕРЖАНИЕ ВСЕГДА НА РУССКОМ.
    """
    if abstract_text:
        text = re.sub(r"\s+", " ", abstract_text).strip()
        sentences = re.split(r"(?<=[.!?])\s+", text)
        sentences = [s.strip() for s in sentences if s.strip()]

        summary_en = " ".join(sentences[:max_sentences]).strip()
        if len(summary_en) > 700:
            summary_en = summary_en[:700].rsplit(" ", 1)[0] + "..."

        summary_ru = _translate_to_ru(summary_en)
        return summary_ru

    return f"Аннотация отсутствует в метаданных OpenAlex. Статья посвящена теме: {title}"


def _extract_pdf_url(work: dict) -> Optional[str]:
    """
    Пытается достать прямую ссылку на PDF.
    Приоритет:
    1) best_oa_location.pdf_url
    2) primary_location.pdf_url
    3) open_access.oa_url (если выглядит как pdf)
    """
    best_pdf = _safe_get(work, "best_oa_location", "pdf_url")
    if best_pdf:
        return best_pdf

    primary_pdf = _safe_get(work, "primary_location", "pdf_url")
    if primary_pdf:
        return primary_pdf

    oa_url = _safe_get(work, "open_access", "oa_url")
    if oa_url and ".pdf" in oa_url.lower():
        return oa_url

    return None


def _authors_to_string(authorships: list) -> str:
    """
    Собирает строку авторов из authorships.
    """
    authors = []
    for a in authorships or []:
        name = _safe_get(a, "author", "display_name")
        if name:
            authors.append(name)

    if not authors:
        return "Не указано"

    if len(authors) <= 6:
        return ", ".join(authors)

    return ", ".join(authors[:6]) + " et al."


def _sanitize_filename(name: str, max_len: int = 180) -> str:
    """
    Чистит имя файла от запрещённых символов.
    """
    name = re.sub(r'[\\/*?:"<>|]+', "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    if len(name) > max_len:
        name = name[:max_len].rstrip()
    return name


# ------------------------------------------
# ПОИСК В OPENALEX
# ------------------------------------------

def search_openalex_articles_with_pdf(
    query: str,
    max_results: int = MAX_RESULTS_DEFAULT,
    per_page: int = 50
) -> List[Dict]:
    """
    Ищет статьи в OpenAlex по текстовому запросу.
    Возвращает только статьи, у которых найден PDF.
    """
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    results: List[Dict] = []
    page = 1

    while len(results) < max_results:
        params = {
            "search": query,
            "per-page": min(per_page, 200),
            "page": page,
            "sort": "relevance_score:desc",
        }

        response = session.get(OPENALEX_WORKS_URL, params=params, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        works = data.get("results", [])
        if not works:
            break

        for work in works:
            pdf_url = _extract_pdf_url(work)
            if not pdf_url:
                continue

            title = work.get("display_name") or "Без названия"
            abstract_text = _reconstruct_abstract(work.get("abstract_inverted_index"))
            language = work.get("language") or _guess_language(title + " " + abstract_text)

            item = {
                "id": work.get("id"),
                "title": title,
                "authors": _authors_to_string(work.get("authorships")),
                "year": work.get("publication_year"),
                "language": language,
                "summary": _build_short_summary(title, abstract_text),
                "pdf_url": pdf_url,
                "doi": work.get("doi"),
                "openalex_url": work.get("id"),
            }
            results.append(item)

            if len(results) >= max_results:
                break

        page += 1

    return results


# ------------------------------------------
# СКАЧИВАНИЕ PDF
# ------------------------------------------

def download_selected_articles(selected_articles: List[Dict], output_folder: str) -> List[Path]:
    """
    Скачивает выбранные статьи в указанную папку.
    """
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    downloaded_files = []

    for idx, article in enumerate(selected_articles, start=1):
        title = article["title"]
        year = article.get("year") or "unknown_year"
        authors = article.get("authors") or "unknown_author"

        short_author = authors.split(",")[0].strip()
        filename = _sanitize_filename(f"{year} - {short_author} - {title}.pdf")
        file_path = output_path / filename

        pdf_url = article["pdf_url"]

        try:
            with session.get(pdf_url, stream=True, timeout=REQUEST_TIMEOUT, allow_redirects=True) as r:
                r.raise_for_status()

                content_type = r.headers.get("Content-Type", "").lower()
                if "pdf" not in content_type and not pdf_url.lower().endswith(".pdf"):
                    # Всё равно пробуем скачать — некоторые репозитории отдают PDF без корректного content-type
                    pass

                with open(file_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)

            downloaded_files.append(file_path)

        except Exception as e:
            print(f"❌ Не удалось скачать: {title}\n   Причина: {e}\n")

    return downloaded_files


# ------------------------------------------
# ИНТЕРАКТИВНЫЙ ИНТЕРФЕЙС ДЛЯ JUPYTER
# ------------------------------------------

def openalex_search_and_download_ui(
    query: str,
    output_folder: str,
    max_results: int = 15
):
    """
    Основная функция:
    1) ищет статьи по запросу
    2) показывает чекбоксы
    3) позволяет скачать выбранные статьи в папку
    """
    results = search_openalex_articles_with_pdf(query=query, max_results=max_results)

    if not results:
        print("Ничего не найдено с доступным PDF.")
        return

    checkboxes = []
    article_boxes = []

    for i, article in enumerate(results, start=1):
        cb = widgets.Checkbox(value=False, description="", indent=False)
        checkboxes.append(cb)

        title_html = f"<b>{i}. {article['title']}</b>"
        meta_html = (
            f"<div style='margin-top:4px;'>"
            f"<b>Авторы:</b> {article['authors']}<br>"
            f"<b>Год:</b> {article['year']}<br>"
            f"<b>Язык:</b> {article['language']}<br>"
            f"<b>PDF:</b> <a href='{article['pdf_url']}' target='_blank'>ссылка</a><br>"
            f"<b>Краткое содержание:</b> {article['summary']}"
            f"</div>"
        )

        box = widgets.HBox([
            cb,
            widgets.HTML(value=f"<div style='padding:6px 0 12px 0;'>{title_html}{meta_html}</div>")
        ])
        article_boxes.append(box)

    select_all_btn = widgets.Button(description="Выбрать все", button_style="")
    unselect_all_btn = widgets.Button(description="Снять все", button_style="")
    download_btn = widgets.Button(description="Скачать выбранные", button_style="success")
    out = widgets.Output()

    def on_select_all(_):
        for cb in checkboxes:
            cb.value = True

    def on_unselect_all(_):
        for cb in checkboxes:
            cb.value = False

    def on_download(_):
        with out:
            clear_output()
            selected = [article for cb, article in zip(checkboxes, results) if cb.value]

            if not selected:
                print("Ничего не выбрано.")
                return

            print(f"Начинаю скачивание: {len(selected)} файл(ов)")
            downloaded = download_selected_articles(selected, output_folder)

            print("\n✅ Скачано:")
            for path in downloaded:
                print(path)

            # Дополнительно сохраним метаданные выбранных статей
            meta_path = Path(output_folder) / "selected_articles_metadata.json"
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(selected, f, ensure_ascii=False, indent=2)

            print(f"\n📝 Метаданные сохранены: {meta_path}")

    select_all_btn.on_click(on_select_all)
    unselect_all_btn.on_click(on_unselect_all)
    download_btn.on_click(on_download)

    controls = widgets.HBox([select_all_btn, unselect_all_btn, download_btn])
    ui = widgets.VBox([
        widgets.HTML(value=f"<h3>Результаты поиска по запросу: {query}</h3>"),
        controls,
        widgets.VBox(article_boxes),
        out
    ])

    display(ui)

In [7]:
# ==========================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ==========================================

openalex_search_and_download_ui(
    query="Zinc–air batteries: from fundamental principles to recent progress and future challenges",
    output_folder=r"H:\Мой диск\BESS\papers",
    max_results=10
)